# JTMT PUMA validator and safe auto-fixer

This notebook validates a scenario PUMA file against its BaseProjection, applies only auditable fixes, and reports unresolved cases.

## Registered rules


| ID | Source | Rule | Severity | Fix |
|---|---|---|---|---|
| S1 | mandatory | Required columns are present | HARD_STOP | none |
| S2 | mandatory | TAZ is a non-null integer; whitespace and trailing `.0` may be cleaned | HARD_STOP | normalize |
| S3 | mandatory | No duplicate TAZ; drop identical extras, report conflicting duplicates | HARD_STOP | deterministic/manual |
| S4 | mandatory | PUMA is integer and code columns are trimmed | ERROR | normalize, REF, or unresolved |
| EX | mandatory | Palestinian rows are excluded from checks and unchanged | SCOPE | none |
| R4 | mandatory | PUMA and BP TAZ sets match after exclusion | HARD_STOP | REF for missing; configured drop for extras |
| R1 | mandatory | Positive metropolitan population has PUMA other than 999 | ERROR | valid REF, sector F-field, then district mode >= threshold |
| R2 | mandatory | Positive population has nonblank DISTRICT | ERROR | REF then PUMA mode >= threshold |
| R3 | mandatory | Arab sectors: nonpositive/missing population has PUMA 999; positive population has PUMA 17-19 | ERROR | deterministic 999, otherwise REF/group |
| C1 | conversation | Sector fallback is Jewish->F3, U_Orthodox->F2, Arab groups->F1; valid REF calibration may override it | ERROR | REF then F field |
| C2 | conversation | One assignment row per TAZ | HARD_STOP | covered by S3 |
| C3 | conversation | Population removed relative to baseline gets PUMA=999 and DISTRICT=999 | ERROR | deterministic |
| C4 | conversation | Newly populated TAZ receives valid PUMA, DISTRICT, and school district | ERROR | normal candidate hierarchy |
| C5 | conversation | School district remains geographic when population is zero | ERROR | retain REF school value |
| C6 | conversation | Unchanged population presence carries the known-good fixed assignment | ERROR | retain REF, subject to mandatory rules |
| C7 | conversation | Positive metropolitan population has DISTRICT other than 999, except documented exceptions | ERROR | REF then PUMA mode |
| C8 | conversation | NaN, zero, and missing TAZ are distinct | HARD_STOP | report separately |
| C9 | conversation | Normalize TAZ consistently and join only on TAZ | HARD_STOP | normalize |
| C10 | conversation | Joins must not multiply rows | HARD_STOP | stop |
| C11 | conversation | BP population remains at source total after joins | HARD_STOP | stop |
| C12 | conversation | Weak or ambiguous evidence remains unresolved with candidates recorded | WARNING | manual |
| C13 | conversation/data | Populated Arab PUMA is 17-19 | ERROR | covered by R3 |
| C14 | conversation/data | Eitan overrides are retained as audit history but cannot override mandatory R1 | AUDIT | no active exception |
| C15 | conversation | Never overwrite inputs | HARD_STOP | separate outputs |
| SD1 | conversation | SCHOOLDISTRICT is a nonblank integer | ERROR | normalize/REF |
| SD2 | conversation | Palestinian school assignments remain unchanged | SCOPE | none |
| SD3 | conversation/data | Positive non-Palestinian metropolitan population has school district other than 999 | ERROR | valid REF/geographic mode |
| SD4 | conversation/data | Zero population does not itself imply school district 999 | ERROR | preserve geographic REF |
| SD5 | conversation/data | Unchanged population presence retains Eitan's fixed school assignment | ERROR | REF |
| SD6 | conversation/data | Newly populated TAZ validates school assignment against populated geographic peers | ERROR | supported geographic mode |
| SD7 | conversation/data | Prefer same AGG_TAZ and municipality; sector/PUMA are filters, not geography | ERROR | supported mode |
| SD8 | conversation/data | Split geography or weak support is unresolved | WARNING | manual |
| SD9 | conversation/data | School code must be in the configured/REF domain | ERROR | REF/group |
| SD10 | conversation | Every school change is audited | AUDIT | change log |


## Configuration

All scenario-specific paths, thresholds, and documented exceptions are defined in the following single cell.

In [1]:

from pathlib import Path

# Edit this cell only when moving the checker to another scenario/project.
CONFIG = {
    "PUMA_PATH": Path("outputs/puma2050_260310_jtmt_09_proposed.csv"),
    "BP_PATH": Path("background_files/BaseProjections2050_260310_jtmt_09.csv"),
    "BASELINE_BP_PATH": Path("background_files/BaseProjections2050_250226_jtmt.xlsx"),
    "REF_PUMA_PATH": Path("background_files/puma2050_fixed.csv"),
    "GEO_PATH": Path("background_files/TAZ_V4_241103_with_geo_info.shp"),
    "OUTPUT_DIR": Path("outputs/puma_checker_run"),
    "MODE_SHARE_THRESHOLD": 0.60,
    "MIN_GROUP_SIZE": 2,
    "MAX_ITERATIONS": 5,
    "DROP_EXTRA_PUMA_TAZ": False,
    "ENCODINGS": ["utf-8-sig", "utf-8", "cp1255"],
    "MANUAL_EXCEPTIONS": {"PUMA": {}, "DISTRICT": {}},
}


## Validation and fix engine

Each business rule is represented by one `Rule` entry with a check function and, where safe, a fix function.

In [2]:

from dataclasses import dataclass
from typing import Callable, Optional
import json
import math
import shutil

import numpy as np
import pandas as pd

PUMA_REQUIRED = ["TAZ", "PUMA", "DISTRICT", "REGION", "SCHOOLDISTRICT"]
BP_REQUIRED = ["TAZ", "sector", "pop", "in_jerusalem_metropolin"]
CODE_COLUMNS = ["PUMA", "DISTRICT", "REGION", "SCHOOLDISTRICT"]
ARAB_SECTORS = {"arab", "arabs_behined_seperation_wall"}


@dataclass
class Rule:
    rule_id: str
    severity: str
    description: str
    check: Callable
    fix: Optional[Callable] = None


def finding(rule_id, severity, taz=None, column=None, old=None, reason="", candidates=""):
    return {
        "rule_id": rule_id, "severity": severity, "TAZ": taz, "column": column,
        "old": old, "reason": reason, "candidates_considered": candidates,
    }


def detect_encoding(path, encodings):
    raw = Path(path).read_bytes()
    for enc in encodings:
        try:
            raw.decode(enc)
        except UnicodeDecodeError:
            continue
        if enc == "utf-8-sig" and not raw.startswith(b"\xef\xbb\xbf"):
            continue
        return enc
    raise UnicodeError(f"Could not decode {path} using {encodings}")


def read_table(path, encodings):
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path), None
    encoding = detect_encoding(path, encodings)
    return pd.read_csv(path, encoding=encoding, dtype=str), encoding


def normalize_integer(series):
    text = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    numeric = pd.to_numeric(text, errors="coerce")
    valid = series.isna() | (numeric.notna() & np.isclose(numeric, np.round(numeric)))
    return numeric.round().astype("Int64"), valid


def sector_norm(series):
    return series.astype("string").str.strip().str.casefold()


def load_geo(path):
    if path is None or not Path(path).exists():
        return None
    path = Path(path)
    if path.suffix.lower() == ".shp":
        import geopandas as gpd
        geo = gpd.read_file(path).drop(columns="geometry", errors="ignore")
    else:
        geo, _ = read_table(path, CONFIG["ENCODINGS"])
    rename = {"Taz_num": "TAZ", "Agg_taz_nu": "AGG_TAZ", "Muni_Heb": "muni_heb"}
    geo = geo.rename(columns={k: v for k, v in rename.items() if k in geo.columns})
    if "TAZ" in geo:
        geo["TAZ"], _ = normalize_integer(geo["TAZ"])
    return geo


def prepare_context(config, puma_df=None, bp_df=None, ref_df=None, baseline_df=None, geo_df=None):
    puma_path = Path(config["PUMA_PATH"])
    if puma_df is None:
        puma_df, input_encoding = read_table(puma_path, config["ENCODINGS"])
    else:
        puma_df, input_encoding = puma_df.copy(), "utf-8"
    if bp_df is None:
        bp_df, _ = read_table(config["BP_PATH"], config["ENCODINGS"])
    if ref_df is None and config.get("REF_PUMA_PATH") and Path(config["REF_PUMA_PATH"]).exists():
        ref_df, _ = read_table(config["REF_PUMA_PATH"], config["ENCODINGS"])
    if baseline_df is None and config.get("BASELINE_BP_PATH") and Path(config["BASELINE_BP_PATH"]).exists():
        baseline_df, _ = read_table(config["BASELINE_BP_PATH"], config["ENCODINGS"])
    if geo_df is None:
        geo_df = load_geo(config.get("GEO_PATH"))
    return {
        "config": config, "puma": puma_df.copy(), "bp": bp_df.copy(),
        "ref": None if ref_df is None else ref_df.copy(),
        "baseline": None if baseline_df is None else baseline_df.copy(),
        "geo": geo_df, "input_columns": list(puma_df.columns),
        "input_encoding": input_encoding, "input_row_order": list(range(len(puma_df))),
        "changes": [], "unresolved": [], "hard_stop": False,
    }


def add_change(ctx, rule_id, taz, column, new, method, confidence):
    mask = ctx["puma"]["TAZ"].eq(taz)
    if not mask.any():
        return False
    old = ctx["puma"].loc[mask, column].iloc[0]
    same = (pd.isna(old) and pd.isna(new)) or (pd.notna(old) and pd.notna(new) and old == new)
    if same:
        return False
    ctx["puma"].loc[mask, column] = new
    ctx["changes"].append({
        "rule_id": rule_id, "TAZ": int(taz), "column": column,
        "old": old, "new": new, "method": method, "confidence": confidence,
    })
    return True


def build_join(ctx):
    p = ctx["puma"].copy()
    b = ctx["bp"].copy()
    if "TAZ" not in p or "TAZ" not in b:
        return pd.DataFrame()
    b["sector_norm"] = sector_norm(b["sector"])
    keep = ["TAZ", "sector", "sector_norm", "pop", "in_jerusalem_metropolin"]
    j = p.merge(b[keep], on="TAZ", how="left", validate="one_to_one")
    if ctx["baseline"] is not None and "TAZ" in ctx["baseline"]:
        bb = ctx["baseline"][["TAZ", "pop"]].rename(columns={"pop": "baseline_pop"}).copy()
        j = j.merge(bb, on="TAZ", how="left", validate="one_to_one")
    if ctx["geo"] is not None and "TAZ" in ctx["geo"]:
        cols = [c for c in ["TAZ", "AGG_TAZ", "muni_heb", "F1", "F2", "F3", "SCHOOLDIST"] if c in ctx["geo"]]
        extra = ctx["geo"][cols].drop_duplicates("TAZ")
        overlap = [c for c in extra.columns if c != "TAZ" and c in j.columns]
        extra = extra.rename(columns={c: f"geo_{c}" for c in overlap})
        j = j.merge(extra, on="TAZ", how="left", validate="one_to_one")
    return j


def exception_value(ctx, column, taz):
    item = ctx["config"].get("MANUAL_EXCEPTIONS", {}).get(column, {}).get(int(taz))
    return None if item is None else int(item["value"])


def ref_value(ctx, taz, column):
    ref = ctx["ref"]
    if ref is None or "TAZ" not in ref or column not in ref:
        return None
    hit = ref.loc[ref["TAZ"].eq(taz), column]
    if len(hit) != 1 or pd.isna(hit.iloc[0]):
        return None
    return int(hit.iloc[0])


def valid_puma_for_row(row, value):
    if value is None:
        return False
    pop = row.get("pop")
    sector = row.get("sector_norm")
    metro = row.get("in_jerusalem_metropolin")
    if sector in ARAB_SECTORS:
        if pd.isna(pop) or pop <= 0:
            return value == 999
        return 17 <= value <= 19
    if pd.notna(pop) and pop > 0 and metro == 1:
        return value != 999
    return True


def valid_district_for_row(ctx, row, value):
    if value is None:
        return False
    taz = row.get("TAZ", row.name)
    if exception_value(ctx, "DISTRICT", taz) == value:
        return True
    return not (pd.notna(row.get("pop")) and row.get("pop") > 0 and row.get("in_jerusalem_metropolin") == 1 and value == 999)


def valid_school_for_row(row, value):
    if value is None:
        return False
    return not (row.get("sector_norm") != "palestinian" and pd.notna(row.get("pop")) and row.get("pop") > 0 and row.get("in_jerusalem_metropolin") == 1 and value == 999)


def mode_candidate(ctx, row, column, group_key, validator, exclude_999=False):
    j = build_join(ctx)
    if group_key not in j or pd.isna(row.get(group_key)):
        return None, "group key unavailable"
    taz = row.get("TAZ", row.name)
    peers = j[j[group_key].eq(row[group_key]) & j.TAZ.ne(taz)].copy()
    if exclude_999:
        peers = peers[peers[column].ne(999)]
    if peers.empty:
        return None, "no valid peers"
    peers = peers[peers.apply(lambda r: validator(r, None if pd.isna(r[column]) else int(r[column])), axis=1)]
    vals = peers[column].dropna().astype(int)
    if vals.empty:
        return None, "no valid peers"
    counts = vals.value_counts()
    value, count = int(counts.index[0]), int(counts.iloc[0])
    share = count / len(vals)
    detail = f"{value}: {count}/{len(vals)} ({share:.1%})"
    if len(vals) < ctx["config"]["MIN_GROUP_SIZE"] or share < ctx["config"]["MODE_SHARE_THRESHOLD"]:
        return None, detail
    return value, detail


def school_mode_candidate(ctx, row):
    j = build_join(ctx)
    taz = row.get("TAZ", row.name)
    peers = j[(j.TAZ.ne(taz)) & (j.sector_norm.ne("palestinian")) & (j["pop"] > 0)].copy()
    agg_col = "AGG_TAZ" if "AGG_TAZ" in j else "geo_AGG_TAZ" if "geo_AGG_TAZ" in j else None
    muni_col = "muni_heb" if "muni_heb" in j else "geo_muni_heb" if "geo_muni_heb" in j else None
    attempts = []
    filters = []
    if agg_col and muni_col and pd.notna(row.get(agg_col)) and pd.notna(row.get(muni_col)):
        filters.append((f"{agg_col}+{muni_col}", (peers[agg_col] == row[agg_col]) & (peers[muni_col] == row[muni_col])))
    if agg_col and pd.notna(row.get(agg_col)):
        filters.append((agg_col, peers[agg_col] == row[agg_col]))
    for label, mask in filters:
        vals = peers.loc[mask, "SCHOOLDISTRICT"].dropna().astype(int)
        vals = vals[vals.ne(999)]
        if vals.empty:
            attempts.append(f"{label}: none")
            continue
        counts = vals.value_counts()
        value, count = int(counts.index[0]), int(counts.iloc[0])
        share = count / len(vals)
        attempts.append(f"{label}: {value}={count}/{len(vals)} ({share:.1%})")
        if len(vals) >= ctx["config"]["MIN_GROUP_SIZE"] and share >= ctx["config"]["MODE_SHARE_THRESHOLD"]:
            return value, "; ".join(attempts)
    return None, "; ".join(attempts) or "geographic group unavailable"


def check_s1(ctx):
    out = []
    for label, frame, required in [("PUMA", ctx["puma"], PUMA_REQUIRED), ("BP", ctx["bp"], BP_REQUIRED)]:
        for col in sorted(set(required) - set(frame.columns)):
            out.append(finding("S1", "HARD_STOP", column=col, reason=f"{label} missing required column"))
    return out


def check_s2(ctx):
    out = []
    for label in ["puma", "bp", "ref", "baseline"]:
        frame = ctx.get(label)
        if frame is None or "TAZ" not in frame:
            continue
        norm, valid = normalize_integer(frame["TAZ"])
        for idx in frame.index[~valid | norm.isna()]:
            out.append(finding("S2", "HARD_STOP", old=frame.at[idx, "TAZ"], reason=f"invalid TAZ in {label}"))
        cleanable = valid & norm.notna() & frame["TAZ"].astype("string").ne(norm.astype("string"))
        for idx in frame.index[cleanable]:
            out.append(finding("S2", "HARD_STOP", taz=int(norm.at[idx]), old=frame.at[idx, "TAZ"], reason=f"cleanable TAZ in {label}"))
    return out


def fix_s2(ctx, findings):
    changed = 0
    for label in ["puma", "bp", "ref", "baseline"]:
        frame = ctx.get(label)
        if frame is None or "TAZ" not in frame:
            continue
        norm, valid = normalize_integer(frame["TAZ"])
        if valid.all() and norm.notna().all():
            old = frame["TAZ"].copy()
            frame["TAZ"] = norm
            changed += int((old.astype("string") != norm.astype("string")).sum())
    return changed


def check_s3(ctx):
    out = []
    if "TAZ" not in ctx["puma"]:
        return out
    for taz, group in ctx["puma"][ctx["puma"].TAZ.duplicated(False)].groupby("TAZ", dropna=False):
        identical = len(group.drop_duplicates()) == 1
        out.append(finding("S3", "HARD_STOP", taz=taz, reason="identical duplicate" if identical else "conflicting duplicate"))
    return out


def fix_s3(ctx, findings):
    conflicts = [f for f in findings if f["reason"] == "conflicting duplicate"]
    before = len(ctx["puma"])
    if not conflicts:
        ctx["puma"] = ctx["puma"].drop_duplicates(keep="first").copy()
    return before - len(ctx["puma"])


def check_s4(ctx):
    out = []
    for col in CODE_COLUMNS:
        if col not in ctx["puma"]:
            continue
        norm, valid = normalize_integer(ctx["puma"][col])
        for idx in ctx["puma"].index[~valid | (norm.isna() if col == "PUMA" else False)]:
            out.append(finding("S4", "ERROR", taz=ctx["puma"].at[idx, "TAZ"], column=col, old=ctx["puma"].at[idx, col], reason="invalid integer code"))
    return out


def fix_s4(ctx, findings):
    changed = 0
    for col in CODE_COLUMNS:
        if col not in ctx["puma"]:
            continue
        old = ctx["puma"][col].copy()
        norm, valid = normalize_integer(old)
        ctx["puma"].loc[valid, col] = norm[valid]
        changed += int((old.astype("string") != ctx["puma"][col].astype("string")).sum())
    return changed


def check_ex(ctx):
    return []


def check_r4(ctx):
    if "TAZ" not in ctx["puma"] or "TAZ" not in ctx["bp"] or "sector" not in ctx["bp"]:
        return []
    bp = ctx["bp"].copy()
    scope = bp.loc[sector_norm(bp.sector).ne("palestinian"), "TAZ"]
    pset, bset = set(ctx["puma"].TAZ.dropna()), set(scope.dropna())
    out = [finding("R4", "HARD_STOP", taz=t, reason="TAZ missing from PUMA") for t in sorted(bset - pset)]
    bp_all = set(bp.TAZ.dropna())
    out += [finding("R4", "HARD_STOP", taz=t, reason="TAZ in PUMA but not BP") for t in sorted(pset - bp_all)]
    return out


def fix_r4(ctx, findings):
    changed = 0
    for f in findings:
        taz = f["TAZ"]
        if f["reason"] == "TAZ missing from PUMA" and ctx["ref"] is not None:
            hit = ctx["ref"][ctx["ref"].TAZ.eq(taz)]
            if len(hit) == 1:
                row = {c: hit.iloc[0].get(c, pd.NA) for c in ctx["puma"].columns}
                ctx["puma"] = pd.concat([ctx["puma"], pd.DataFrame([row])], ignore_index=True)
                changed += 1
        elif f["reason"] == "TAZ in PUMA but not BP" and ctx["config"]["DROP_EXTRA_PUMA_TAZ"]:
            ctx["puma"] = ctx["puma"][ctx["puma"].TAZ.ne(taz)].copy()
            changed += 1
    return changed


def check_r1(ctx):
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian":
            continue
        if pd.notna(row.pop) and row.pop > 0 and row.in_jerusalem_metropolin == 1 and row.PUMA == 999:
            if exception_value(ctx, "PUMA", row.TAZ) != 999:
                out.append(finding("R1", "ERROR", row.TAZ, "PUMA", row.PUMA, "positive metropolitan population has PUMA 999"))
    return out


def fix_r1(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        rv = ref_value(ctx, f["TAZ"], "PUMA")
        if valid_puma_for_row(row, rv):
            changed += add_change(ctx, "R1", f["TAZ"], "PUMA", rv, "REF_PUMA same TAZ", 1.0)
            continue
        field = {"jewish": "F3", "u_orthodox": "F2", "arab": "F1", "arabs_behined_seperation_wall": "F1"}.get(row.sector_norm)
        if field:
            field = field if field in row.index else f"geo_{field}"
            expected = row.get(field)
            if pd.notna(expected) and valid_puma_for_row(row, int(expected)):
                changed += add_change(ctx, "R1", f["TAZ"], "PUMA", int(expected), f"sector rule {field}", 1.0)
                continue
        value, detail = mode_candidate(ctx, row, "PUMA", "DISTRICT", valid_puma_for_row, True)
        if value is not None:
            changed += add_change(ctx, "R1", f["TAZ"], "PUMA", value, f"district mode; {detail}", ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_r2(ctx):
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian":
            continue
        if pd.notna(row.pop) and row.pop > 0 and pd.isna(row.DISTRICT):
            out.append(finding("R2", "ERROR", row.TAZ, "DISTRICT", row.DISTRICT, "positive population has blank district"))
    return out


def fix_r2(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        rv = ref_value(ctx, f["TAZ"], "DISTRICT")
        if valid_district_for_row(ctx, row, rv):
            changed += add_change(ctx, "R2", f["TAZ"], "DISTRICT", rv, "REF_PUMA same TAZ", 1.0)
            continue
        value, detail = mode_candidate(ctx, row, "DISTRICT", "PUMA", lambda r, v: valid_district_for_row(ctx, r, v))
        if value is not None:
            changed += add_change(ctx, "R2", f["TAZ"], "DISTRICT", value, f"PUMA mode; {detail}", ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_r3(ctx):
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm not in ARAB_SECTORS:
            continue
        if pd.isna(row.pop) or row.pop <= 0:
            if row.PUMA != 999:
                out.append(finding("R3", "ERROR", row.TAZ, "PUMA", row.PUMA, "Arab nonpositive/missing population requires 999"))
        elif not 17 <= row.PUMA <= 19:
            out.append(finding("R3", "ERROR", row.TAZ, "PUMA", row.PUMA, "populated Arab PUMA must be 17-19"))
    return out


def fix_r3(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        if pd.isna(row["pop"]) or row["pop"] <= 0:
            changed += add_change(ctx, "R3", f["TAZ"], "PUMA", 999, "deterministic no-population rule", 1.0)
            continue
        rv = ref_value(ctx, f["TAZ"], "PUMA")
        if valid_puma_for_row(row, rv):
            changed += add_change(ctx, "R3", f["TAZ"], "PUMA", rv, "REF_PUMA same TAZ", 1.0)
        else:
            value, detail = mode_candidate(ctx, row, "PUMA", "DISTRICT", valid_puma_for_row, True)
            if value is not None:
                changed += add_change(ctx, "R3", f["TAZ"], "PUMA", value, f"Arab district mode; {detail}", ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_c1(ctx):
    if ctx["geo"] is None:
        return []
    out = []
    for _, row in build_join(ctx).iterrows():
        if row.sector_norm == "palestinian" or pd.isna(row["pop"]) or row["pop"] <= 0:
            continue
        field = {"jewish": "F3", "u_orthodox": "F2", "arab": "F1", "arabs_behined_seperation_wall": "F1"}.get(row.sector_norm)
        if not field:
            continue
        if row.sector_norm in ARAB_SECTORS and 17 <= row.PUMA <= 19:
            continue  # R3/C13 calibration supersedes legacy F1 values such as 16.
        field = field if field in row.index else f"geo_{field}"
        expected = row.get(field)
        if pd.notna(expected) and row.PUMA != int(expected):
            rv = ref_value(ctx, row.TAZ, "PUMA")
            calibrated_ref_is_valid = rv == row.PUMA and valid_puma_for_row(row, rv)
            if not calibrated_ref_is_valid:
                out.append(finding("C1", "ERROR", row.TAZ, "PUMA", row.PUMA, f"sector rule expects {int(expected)}"))
    return out


def fix_c1(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        rv = ref_value(ctx, f["TAZ"], "PUMA")
        if valid_puma_for_row(row, rv):
            changed += add_change(ctx, "C1", f["TAZ"], "PUMA", rv, "REF calibration", 1.0)
            continue
        field = {"jewish": "F3", "u_orthodox": "F2", "arab": "F1", "arabs_behined_seperation_wall": "F1"}.get(row.sector_norm)
        if field:
            field = field if field in row.index else f"geo_{field}"
            expected = row.get(field)
            if pd.notna(expected) and valid_puma_for_row(row, int(expected)):
                changed += add_change(ctx, "C1", f["TAZ"], "PUMA", int(expected), f"sector rule {field}", 1.0)
    return changed


def check_c3(ctx):
    if ctx["baseline"] is None:
        return []
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian" or pd.isna(row.baseline_pop):
            continue
        if row.baseline_pop > 0 and (pd.isna(row.pop) or row.pop <= 0):
            for col in ["PUMA", "DISTRICT"]:
                if getattr(row, col) != 999:
                    out.append(finding("C3", "ERROR", row.TAZ, col, getattr(row, col), "population removed; code must be 999"))
    return out


def fix_c3(ctx, findings):
    return sum(add_change(ctx, "C3", f["TAZ"], f["column"], 999, "deterministic removed-population rule", 1.0) for f in findings)


def check_c4(ctx):
    if ctx["baseline"] is None:
        return []
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian" or pd.isna(row.baseline_pop):
            continue
        if row.baseline_pop <= 0 and pd.notna(row.pop) and row.pop > 0:
            if row.PUMA == 999:
                out.append(finding("C4", "ERROR", row.TAZ, "PUMA", row.PUMA, "new population needs PUMA"))
            if pd.isna(row.DISTRICT) or row.DISTRICT == 999:
                out.append(finding("C4", "ERROR", row.TAZ, "DISTRICT", row.DISTRICT, "new population needs district"))
            if pd.isna(row.SCHOOLDISTRICT) or row.SCHOOLDISTRICT == 999:
                out.append(finding("C4", "ERROR", row.TAZ, "SCHOOLDISTRICT", row.SCHOOLDISTRICT, "new population needs school district"))
    return out


def fix_c4(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        col = f["column"]
        if col == "PUMA":
            rv = ref_value(ctx, f["TAZ"], col)
            if valid_puma_for_row(row, rv):
                changed += add_change(ctx, "C4", f["TAZ"], col, rv, "REF_PUMA same TAZ", 1.0)
        elif col == "DISTRICT":
            rv = ref_value(ctx, f["TAZ"], col)
            if valid_district_for_row(ctx, row, rv):
                changed += add_change(ctx, "C4", f["TAZ"], col, rv, "REF_PUMA same TAZ", 1.0)
        else:
            value, detail = school_mode_candidate(ctx, row)
            if value is None:
                rv = ref_value(ctx, f["TAZ"], col)
                value = rv if valid_school_for_row(row, rv) else None
                detail = "REF_PUMA same TAZ"
            if value is not None:
                changed += add_change(ctx, "C4", f["TAZ"], col, value, detail, ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_c5(ctx):
    if ctx["baseline"] is None or ctx["ref"] is None:
        return []
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian" or not (pd.isna(row.pop) or row.pop <= 0):
            continue
        rv = ref_value(ctx, row.TAZ, "SCHOOLDISTRICT")
        if rv is not None and row.SCHOOLDISTRICT != rv:
            out.append(finding("C5", "ERROR", row.TAZ, "SCHOOLDISTRICT", row.SCHOOLDISTRICT, f"zero population retains REF school {rv}"))
    return out


def fix_c5(ctx, findings):
    return sum(add_change(ctx, "C5", f["TAZ"], "SCHOOLDISTRICT", ref_value(ctx, f["TAZ"], "SCHOOLDISTRICT"), "retain geographic REF", 1.0) for f in findings)


def check_c6(ctx):
    return []  # Enforced through REF priority and explicit exception rule C14.


def check_c7(ctx):
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm == "palestinian":
            continue
        if pd.notna(row.pop) and row.pop > 0 and row.in_jerusalem_metropolin == 1 and row.DISTRICT == 999:
            if exception_value(ctx, "DISTRICT", row.TAZ) != 999:
                out.append(finding("C7", "ERROR", row.TAZ, "DISTRICT", row.DISTRICT, "positive metropolitan population has district 999"))
    return out


def fix_c7(ctx, findings):
    return fix_r2(ctx, findings)


def check_c8(ctx):
    return []


def check_c9(ctx):
    return []


def check_c10(ctx):
    out = []
    if "TAZ" in ctx["puma"] and "TAZ" in ctx["bp"]:
        if len(ctx["puma"].merge(ctx["bp"][["TAZ"]], on="TAZ", how="left")) != len(ctx["puma"]):
            out.append(finding("C10", "HARD_STOP", reason="join multiplies PUMA rows"))
    return out


def check_c11(ctx):
    out = []
    if "pop" in ctx["bp"] and not ctx["bp"]["TAZ"].duplicated().any():
        before = pd.to_numeric(ctx["bp"]["pop"], errors="coerce").sum(min_count=1)
        joined = build_join(ctx)
        after = pd.to_numeric(joined["pop"], errors="coerce").sum(min_count=1)
        matched = set(ctx["puma"].TAZ) >= set(ctx["bp"].TAZ)
        if matched and not np.isclose(before, after, equal_nan=True):
            out.append(finding("C11", "HARD_STOP", reason=f"population total changed: {before} -> {after}"))
    return out


def check_c12(ctx):
    return []


def check_c13(ctx):
    return []  # Duplicate of R3, retained in registry for traceability.


def check_c14(ctx):
    out = []
    for col, values in ctx["config"].get("MANUAL_EXCEPTIONS", {}).items():
        for taz, spec in values.items():
            hit = ctx["puma"].loc[ctx["puma"].TAZ.eq(taz), col]
            if len(hit) == 1 and hit.iloc[0] != spec["value"]:
                out.append(finding("C14", "ERROR", taz, col, hit.iloc[0], spec["reason"]))
    return out


def fix_c14(ctx, findings):
    return sum(add_change(ctx, "C14", f["TAZ"], f["column"], exception_value(ctx, f["column"], f["TAZ"]), "configured manual exception", 1.0) for f in findings)


def check_c15(ctx):
    output = Path(ctx["config"]["OUTPUT_DIR"]) / f"{Path(ctx['config']['PUMA_PATH']).stem}_fixed.csv"
    return [finding("C15", "HARD_STOP", reason="output path equals input path")] if output.resolve() == Path(ctx["config"]["PUMA_PATH"]).resolve() else []


def check_sd1(ctx):
    out = []
    if "SCHOOLDISTRICT" not in ctx["puma"]:
        return out
    norm, valid = normalize_integer(ctx["puma"]["SCHOOLDISTRICT"])
    for idx in ctx["puma"].index[~valid | norm.isna()]:
        out.append(finding("SD1", "ERROR", ctx["puma"].at[idx, "TAZ"], "SCHOOLDISTRICT", ctx["puma"].at[idx, "SCHOOLDISTRICT"], "invalid school code"))
    return out


def fix_sd1(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        rv = ref_value(ctx, f["TAZ"], "SCHOOLDISTRICT")
        if valid_school_for_row(row, rv):
            changed += add_change(ctx, "SD1", f["TAZ"], "SCHOOLDISTRICT", rv, "REF_PUMA same TAZ", 1.0)
    return changed


def check_sd2(ctx):
    return []


def check_sd3(ctx):
    out = []
    for row in build_join(ctx).itertuples(index=False):
        if row.sector_norm != "palestinian" and pd.notna(row.pop) and row.pop > 0 and row.in_jerusalem_metropolin == 1 and row.SCHOOLDISTRICT == 999:
            out.append(finding("SD3", "ERROR", row.TAZ, "SCHOOLDISTRICT", 999, "positive metropolitan population has school district 999"))
    return out


def fix_sd3(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        row = j.loc[f["TAZ"]]
        rv = ref_value(ctx, f["TAZ"], "SCHOOLDISTRICT")
        if valid_school_for_row(row, rv):
            changed += add_change(ctx, "SD3", f["TAZ"], "SCHOOLDISTRICT", rv, "REF_PUMA same TAZ", 1.0)
            continue
        value, detail = school_mode_candidate(ctx, row)
        if value is not None:
            changed += add_change(ctx, "SD3", f["TAZ"], "SCHOOLDISTRICT", value, detail, ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_sd4(ctx):
    return check_c5(ctx)


def fix_sd4(ctx, findings):
    return fix_c5(ctx, findings)


def check_sd5(ctx):
    return []  # REF preservation is implemented by C5/C6 and candidate priority.


def check_sd6(ctx):
    if ctx["baseline"] is None:
        return []
    out = []
    for _, row in build_join(ctx).iterrows():
        if row.sector_norm == "palestinian" or pd.isna(row.get("baseline_pop")):
            continue
        if row.baseline_pop <= 0 and pd.notna(row["pop"]) and row["pop"] > 0:
            value, detail = school_mode_candidate(ctx, row)
            if value is not None and row.SCHOOLDISTRICT != value:
                out.append(finding("SD6", "ERROR", row.TAZ, "SCHOOLDISTRICT", row.SCHOOLDISTRICT, f"newly populated geographic mode is {value}", detail))
    return out


def fix_sd6(ctx, findings):
    changed = 0
    j = build_join(ctx).set_index("TAZ")
    for f in findings:
        value, detail = school_mode_candidate(ctx, j.loc[f["TAZ"]])
        if value is not None:
            changed += add_change(ctx, "SD6", f["TAZ"], "SCHOOLDISTRICT", value, detail, ctx["config"]["MODE_SHARE_THRESHOLD"])
    return changed


def check_sd7(ctx): return []
def check_sd8(ctx): return []


def check_sd9(ctx):
    if ctx["ref"] is None or "SCHOOLDISTRICT" not in ctx["ref"]:
        return []
    domain = set(pd.to_numeric(ctx["ref"]["SCHOOLDISTRICT"], errors="coerce").dropna().astype(int))
    out = []
    for row in ctx["puma"].itertuples(index=False):
        if pd.notna(row.SCHOOLDISTRICT) and int(row.SCHOOLDISTRICT) not in domain:
            out.append(finding("SD9", "ERROR", row.TAZ, "SCHOOLDISTRICT", row.SCHOOLDISTRICT, f"outside REF domain {sorted(domain)}"))
    return out


def fix_sd9(ctx, findings):
    return fix_sd1(ctx, findings)


def check_sd10(ctx): return []


RULES = [
    Rule("S1", "HARD_STOP", "required columns", check_s1),
    Rule("S2", "HARD_STOP", "TAZ normalization", check_s2, fix_s2),
    Rule("S3", "HARD_STOP", "duplicate TAZ", check_s3, fix_s3),
    Rule("S4", "ERROR", "integer code normalization", check_s4, fix_s4),
    Rule("EX", "SCOPE", "Palestinian exclusion", check_ex),
    Rule("R4", "HARD_STOP", "TAZ set equality", check_r4, fix_r4),
    Rule("R1", "ERROR", "positive metropolitan PUMA", check_r1, fix_r1),
    Rule("R2", "ERROR", "positive population district", check_r2, fix_r2),
    Rule("R3", "ERROR", "Arab PUMA", check_r3, fix_r3),
    Rule("C1", "ERROR", "sector F-field assignment", check_c1, fix_c1),
    Rule("C2", "HARD_STOP", "one row per TAZ", check_s3, fix_s3),
    Rule("C3", "ERROR", "removed population", check_c3, fix_c3),
    Rule("C4", "ERROR", "new population", check_c4, fix_c4),
    Rule("C5", "ERROR", "geographic school when empty", check_c5, fix_c5),
    Rule("C6", "ERROR", "carry fixed assignment", check_c6),
    Rule("C7", "ERROR", "positive metropolitan district", check_c7, fix_c7),
    Rule("C8", "HARD_STOP", "NaN/zero/missing distinction", check_c8),
    Rule("C9", "HARD_STOP", "TAZ join key", check_c9),
    Rule("C10", "HARD_STOP", "join cardinality", check_c10),
    Rule("C11", "HARD_STOP", "population total", check_c11),
    Rule("C12", "WARNING", "manual ambiguity", check_c12),
    Rule("C13", "ERROR", "Arab calibration", check_c13),
    Rule("C14", "ERROR", "manual exceptions", check_c14, fix_c14),
    Rule("C15", "HARD_STOP", "no overwrite", check_c15),
    Rule("SD1", "ERROR", "school integer", check_sd1, fix_sd1),
    Rule("SD2", "SCOPE", "Palestinian school exclusion", check_sd2),
    Rule("SD3", "ERROR", "positive metropolitan school", check_sd3, fix_sd3),
    Rule("SD4", "ERROR", "school remains geographic", check_sd4, fix_sd4),
    Rule("SD5", "ERROR", "retain fixed school", check_sd5),
    Rule("SD6", "ERROR", "new-population school geography", check_sd6, fix_sd6),
    Rule("SD7", "ERROR", "school geographic grouping", check_sd7),
    Rule("SD8", "WARNING", "split school geography", check_sd8),
    Rule("SD9", "ERROR", "school code domain", check_sd9, fix_sd9),
    Rule("SD10", "AUDIT", "school audit trail", check_sd10),
]


def normalize_inputs(ctx):
    for label in ["puma", "bp", "ref", "baseline"]:
        frame = ctx.get(label)
        if frame is None:
            continue
        if "TAZ" in frame:
            frame["TAZ"], _ = normalize_integer(frame["TAZ"])
        if label == "puma":
            for col in CODE_COLUMNS:
                if col in frame:
                    frame[col], _ = normalize_integer(frame[col])
        elif label == "ref":
            for col in CODE_COLUMNS:
                if col in frame:
                    frame[col], _ = normalize_integer(frame[col])
        else:
            if "pop" in frame:
                frame["pop"] = pd.to_numeric(frame["pop"], errors="coerce")
            if "in_jerusalem_metropolin" in frame:
                frame["in_jerusalem_metropolin"] = pd.to_numeric(frame["in_jerusalem_metropolin"], errors="coerce")


def run_checker(config, puma_df=None, bp_df=None, ref_df=None, baseline_df=None, geo_df=None, write_outputs=True):
    ctx = prepare_context(config, puma_df, bp_df, ref_df, baseline_df, geo_df)
    before_original = ctx["puma"].copy(deep=True)
    before_findings = []

    # Structural checks operate before normalization so malformed source values remain auditable.
    for rule in RULES[:6]:
        found = rule.check(ctx)
        before_findings.extend(found)
        if found and rule.fix:
            rule.fix(ctx, found)
    normalize_inputs(ctx)

    remaining_structural = []
    for rule in RULES[:6]:
        remaining_structural.extend(rule.check(ctx))
    if any(f["severity"] == "HARD_STOP" for f in remaining_structural):
        ctx["hard_stop"] = True
    else:
        for iteration in range(config["MAX_ITERATIONS"]):
            changed = 0
            for rule in RULES[6:]:
                found = rule.check(ctx)
                if iteration == 0:
                    before_findings.extend(found)
                if found and rule.fix:
                    changed += rule.fix(ctx, found)
            normalize_inputs(ctx)
            if changed == 0:
                break

    after_findings = []
    for rule in RULES:
        if ctx["hard_stop"] and rule.rule_id in {"R1", "R2", "R3"}:
            continue
        after_findings.extend(rule.check(ctx))

    before_df = pd.DataFrame(before_findings)
    after_df = pd.DataFrame(after_findings)
    changes_df = pd.DataFrame(ctx["changes"], columns=["rule_id", "TAZ", "column", "old", "new", "method", "confidence"])
    unresolved_df = after_df.copy()
    if not unresolved_df.empty:
        unresolved_df["unresolved_reason"] = unresolved_df["reason"]

    ids = [r.rule_id for r in RULES]
    summary = pd.DataFrame({"rule_id": ids})
    for name, frame in [("found", before_df), ("unresolved", after_df)]:
        counts = frame.groupby("rule_id").size() if not frame.empty else pd.Series(dtype=int)
        summary[name] = summary.rule_id.map(counts).fillna(0).astype(int)
    fixed_counts = changes_df.groupby("rule_id").size() if not changes_df.empty else pd.Series(dtype=int)
    summary["fixed"] = summary.rule_id.map(fixed_counts).fillna(0).astype(int)
    summary["severity"] = summary.rule_id.map({r.rule_id: r.severity for r in RULES})

    unresolved_count = len(after_df)
    if ctx["hard_stop"] or any(f["severity"] in {"HARD_STOP", "ERROR"} for f in after_findings):
        status = f"FAIL ({unresolved_count} unresolved)"
    elif unresolved_count:
        status = "PASS WITH WARNINGS"
    else:
        status = "PASS"

    outputs = {}
    if write_outputs:
        outdir = Path(config["OUTPUT_DIR"])
        outdir.mkdir(parents=True, exist_ok=True)
        fixed_path = outdir / f"{Path(config['PUMA_PATH']).stem}_fixed.csv"
        change_path = outdir / "change_log.csv"
        report_path = outdir / "puma_report.xlsx"
        output = ctx["puma"][ctx["input_columns"]].copy()
        for col in CODE_COLUMNS + ["TAZ"]:
            if col in output:
                output[col] = output[col].astype("Int64")
        encoding = ctx["input_encoding"] or "utf-8"
        output.to_csv(fixed_path, index=False, encoding=encoding)
        changes_df.to_csv(change_path, index=False, encoding="utf-8-sig")
        with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
            pd.DataFrame({"Status": [status], "Input": [str(config["PUMA_PATH"])], "Rows": [len(output)]}).to_excel(writer, sheet_name="Summary", index=False, startrow=0)
            summary.to_excel(writer, sheet_name="Summary", index=False, startrow=4)
            before_df.to_excel(writer, sheet_name="Findings_before", index=False)
            after_df.to_excel(writer, sheet_name="Findings_after", index=False)
            changes_df.to_excel(writer, sheet_name="Changes", index=False)
            unresolved_df.to_excel(writer, sheet_name="Unresolved", index=False)
        outputs = {"fixed": fixed_path, "changes": change_path, "report": report_path}
    print(status)
    return {"status": status, "context": ctx, "before": before_df, "after": after_df, "changes": changes_df, "summary": summary, "outputs": outputs, "original": before_original}


## Run the configured project

The input is never overwritten. Outputs are written beneath the configured output directory.

In [3]:

result = run_checker(CONFIG)
display(result["summary"])
display(result["changes"])
if not result["after"].empty:
    display(result["after"])
result["outputs"]


PASS


,rule_id,found,unresolved,fixed,severity
0,S1,0,0,0,HARD_STOP
1,S2,0,0,0,HARD_STOP
2,S3,0,0,0,HARD_STOP
3,S4,0,0,0,ERROR
4,EX,0,0,0,SCOPE
5,R4,0,0,0,HARD_STOP
6,R1,5,0,5,ERROR
7,R2,0,0,5,ERROR
8,R3,0,0,0,ERROR
9,C1,0,0,0,ERROR


,rule_id,TAZ,column,old,new,method,confidence
0,R1,1405,PUMA,999,3,sector rule F3,1.0
1,R1,5503,PUMA,999,7,sector rule F2,1.0
2,R1,7062,PUMA,999,9,sector rule F3,1.0
3,R1,8484,PUMA,999,15,sector rule F2,1.0
4,R1,8485,PUMA,999,15,sector rule F2,1.0
5,R2,1405,DISTRICT,999,3,PUMA mode; 3: 32/32 (100.0%),0.6
6,R2,5503,DISTRICT,999,2,PUMA mode; 2: 17/17 (100.0%),0.6
7,R2,7062,DISTRICT,999,5,PUMA mode; 5: 31/31 (100.0%),0.6
8,R2,8484,DISTRICT,999,6,PUMA mode; 6: 56/56 (100.0%),0.6
9,R2,8485,DISTRICT,999,6,PUMA mode; 6: 57/57 (100.0%),0.6


{'fixed': WindowsPath('outputs/puma_checker_run/puma2050_260310_jtmt_09_proposed_fixed.csv'),
 'changes': WindowsPath('outputs/puma_checker_run/change_log.csv'),
 'report': WindowsPath('outputs/puma_checker_run/puma_report.xlsx')}

## Synthetic regression tests

The tests exercise structural cleanup, each deterministic core fix, Palestinian exclusion, missing-TAZ recovery, school logic, and clean-file identity.

In [4]:

def synthetic_clean():
    puma = pd.DataFrame({
        "TAZ": range(1, 21),
        "PUMA": [3, 3, 17, 18, 7, 9, 15, 3, 7, 9, 17, 18, 3, 7, 9, 15, 3, 7, 9, 999],
        "DISTRICT": [3, 3, 1, 1, 2, 5, 6, 3, 2, 5, 1, 1, 3, 2, 5, 6, 3, 2, 5, 999],
        "REGION": [1] * 20,
        "SCHOOLDISTRICT": [2, 2, 1, 1, 6, 9, 16, 2, 6, 9, 1, 1, 2, 6, 9, 16, 2, 6, 9, 999],
        "AGG_TAZ": [1, 1, 2, 2, 3, 4, 5, 1, 3, 4, 2, 2, 1, 3, 4, 5, 1, 3, 4, 20],
    })
    bp = pd.DataFrame({
        "TAZ": range(1, 21),
        "sector": ["Jewish", "Jewish", "Arab", "Arab", "U_Orthodox", "Jewish", "U_Orthodox", "Jewish", "U_Orthodox", "Jewish", "Arab", "Arab", "Jewish", "U_Orthodox", "Jewish", "U_Orthodox", "Jewish", "U_Orthodox", "Jewish", "Palestinian"],
        "pop": [100] * 19 + [100],
        "in_jerusalem_metropolin": [1] * 20,
    })
    baseline = bp.copy()
    return puma, bp, baseline


test_config = dict(CONFIG)
test_config.update({
    "PUMA_PATH": Path("synthetic.csv"), "BP_PATH": Path("synthetic_bp.csv"),
    "REF_PUMA_PATH": None, "BASELINE_BP_PATH": None, "GEO_PATH": None,
    "OUTPUT_DIR": Path("outputs/_synthetic_not_written"), "MANUAL_EXCEPTIONS": {},
})

clean, clean_bp, clean_base = synthetic_clean()
clean_result = run_checker(test_config, clean, clean_bp, clean.copy(), clean_base, None, write_outputs=False)
assert clean_result["status"] == "PASS", clean_result["after"]
pd.testing.assert_frame_equal(clean_result["context"]["puma"].reset_index(drop=True), clean.astype({c: "Int64" for c in clean.columns}).reset_index(drop=True), check_dtype=False)

# One focused injection per core rule; excluded Palestinian TAZ 20 must never change.
cases = []

p, b, base = synthetic_clean(); p.loc[p.TAZ.eq(1), "PUMA"] = 999; cases.append(("R1", p, b, base, "PUMA", 1))
p, b, base = synthetic_clean(); p.loc[p.TAZ.eq(1), "DISTRICT"] = pd.NA; cases.append(("R2", p, b, base, "DISTRICT", 1))
p, b, base = synthetic_clean(); b.loc[b.TAZ.eq(3), "pop"] = 0; p.loc[p.TAZ.eq(3), "PUMA"] = 17; cases.append(("R3", p, b, base, "PUMA", 3))
p, b, base = synthetic_clean(); p.loc[p.TAZ.eq(3), "PUMA"] = 16; cases.append(("R3", p, b, base, "PUMA", 3))
p, b, base = synthetic_clean(); p.loc[p.TAZ.eq(1), "SCHOOLDISTRICT"] = 999; cases.append(("SD3", p, b, base, "SCHOOLDISTRICT", 1))
p, b, base = synthetic_clean(); base.loc[base.TAZ.eq(2), "pop"] = 0; p.loc[p.TAZ.eq(2), "SCHOOLDISTRICT"] = 5; cases.append(("SD6", p, b, base, "SCHOOLDISTRICT", 2))

for expected_rule, p, b, base, column, taz in cases:
    ref = clean.copy()
    res = run_checker(test_config, p, b, ref, base, None, write_outputs=False)
    assert expected_rule in set(res["before"].rule_id), (expected_rule, res["before"])
    assert res["after"].empty, (expected_rule, res["after"])

# Safe TAZ cleaning and identical duplicate removal.
p, b, base = synthetic_clean(); p["TAZ"] = p.TAZ.astype(str); p.loc[0, "TAZ"] = " 1.0 "; p = pd.concat([p, p.iloc[[1]]], ignore_index=True)
res = run_checker(test_config, p, b, clean.copy(), base, None, write_outputs=False)
assert len(res["context"]["puma"]) == 20 and res["after"].empty

# Missing PUMA TAZ is copied from REF; Palestinian row remains byte-for-value equivalent at field level.
p, b, base = synthetic_clean(); p = p[p.TAZ.ne(19)].copy(); palestinian_before = p.loc[p.TAZ.eq(20)].copy()
res = run_checker(test_config, p, b, clean.copy(), base, None, write_outputs=False)
assert 19 in set(res["context"]["puma"].TAZ) and res["after"].empty
pd.testing.assert_frame_equal(palestinian_before.reset_index(drop=True), res["context"]["puma"].loc[res["context"]["puma"].TAZ.eq(20), palestinian_before.columns].reset_index(drop=True), check_dtype=False)

print("Synthetic tests: PASS")


PASS


PASS


PASS


PASS


PASS


PASS


PASS


PASS
PASS
Synthetic tests: PASS
